# 장기 실행 Agent를 위한 Context Editing과 Memory

여러 session에 걸쳐 실행되거나 장기 작업을 처리하는 AI agent는 두 가지 핵심 어려움을 겪습니다. 대화 사이에 학습한 패턴을 잃고, 긴 상호작용 중 context window가 가득 찹니다.

이 cookbook은 Claude의 memory tool과 context editing 기능으로 이러한 어려움을 해결하는 방법을 보여줍니다.


## 목차

1. [소개: Memory가 중요한 이유](#introduction)
2. [사용 사례](#use-cases)
3. [빠른 시작 예제](#quick-start)
4. [동작 방식](#how-it-works)
5. [Code Review Assistant Demo](#demo)
6. [실제 애플리케이션](#real-world)
7. [모범 사례](#best-practices)


## 사전 요구사항

**필요한 지식:**
- Python 기초(함수, class, async/await 기초)
- REST API와 JSON에 대한 기본 이해

**필요한 도구:**
- Python 3.10 이상
- Anthropic API key ([여기에서 발급](https://console.anthropic.com/))

**권장 사항:**
- 동시성 프로그래밍 개념(thread, async)에 대한 이해
- LLM의 context window에 대한 기본 이해

## 설정

### VSCode 사용자

```bash
# 1. 가상 환경 생성
python -m venv .venv

# 2. 활성화
source .venv/bin/activate  # macOS/Linux
# 또는: .venv\Scripts\activate  # Windows

# 3. Dependencies 설치
pip install -r requirements.txt

# 4. VSCode: .venv를 kernel로 선택(오른쪽 위)
```

### API Key

```bash
cp .env.example .env
# .env를 편집하고 ANTHROPIC_API_KEY 추가
```

API key 발급: https://console.anthropic.com/


## 1. 소개: Memory가 중요한 이유 {#introduction}

이 cookbook은 [Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)에서 설명한 context engineering 패턴의 실제 구현을 보여줍니다. 해당 글은 context가 유한한 리소스인 이유, attention budget의 동작 방식, 효과적인 agent 구축 전략을 다룹니다. 여기서는 그 기법이 실제로 동작하는 모습을 볼 수 있습니다.

### 문제

Large language model에는 유한한 context window가 있습니다(Claude 4 모델 계열은 200k token). 커 보이지만 여러 어려움이 생깁니다.

- **Context 한도**: 긴 대화나 복잡한 작업은 사용 가능한 context를 초과할 수 있습니다.
- **계산 비용**: 큰 context를 처리하는 것은 비용이 큽니다. Attention mechanism은 제곱에 비례하여 커집니다.
- **반복 패턴**: 대화마다 비슷한 작업을 할 때 context를 매번 다시 설명해야 합니다.
- **정보 손실**: Context가 가득 차면 앞선 중요한 정보를 잃습니다.

### 해결책

Claude 4 모델은 강력한 context 관리 기능을 도입합니다.

1. **Memory Tool** (`memory_20250818`): 대화 간 학습을 가능하게 합니다.
   - Claude는 배운 내용을 향후 참조를 위해 기록할 수 있습니다.
   - `/memories` 디렉터리 아래의 파일 기반 시스템
   - Client-side 구현으로 완전한 제어 가능

2. **Context Editing**: 두 전략으로 context를 자동 관리합니다.
   - **Tool use clearing** (`clear_tool_uses_20250919`): context가 커지면 이전 tool 결과를 지웁니다.
   - **Thinking management** (`clear_thinking_20251015`): extended thinking block을 관리합니다(thinking 활성화 필요).
   - 설정 가능한 trigger와 보존 정책

### 이점

**시간이 지나면서 특정 작업을 더 잘하는** AI agent를 구축하세요.

- **Session 1**: Claude가 문제를 해결하고 패턴을 기록합니다.
- **Session 2**: Claude가 학습한 패턴을 즉시 적용합니다(더 빠릅니다!).
- **긴 session**: Context editing으로 대화를 관리 가능한 상태로 유지합니다.

사람처럼 메모를 작성하고 다시 참조할 notebook을 Claude에 준다고 생각하세요.

### 학습할 내용

이 cookbook을 마치면 다음을 할 수 있습니다.
- 대화 간 학습을 위한 memory tool **구현**
- 장기 실행 session을 관리하는 context editing **설정**
- Memory 보안과 구성에 관한 모범 사례 **적용**


## 1. 소개: Memory가 중요한 이유 {#introduction}

이 cookbook은 [Effective context engineering for AI agents](https://www.anthropic.com/engineering/effective-context-engineering-for-ai-agents)에서 설명한 context engineering 패턴의 실제 구현을 보여줍니다. 해당 글은 context가 유한한 리소스인 이유, attention budget의 동작 방식, 효과적인 agent 구축 전략을 다룹니다. 여기서는 그 기법이 실제로 동작하는 모습을 볼 수 있습니다.

### 문제

Large language model에는 유한한 context window가 있습니다(Claude 4는 200k token). 커 보이지만 여러 어려움이 생깁니다.

- **Context 한도**: 긴 대화나 복잡한 작업은 사용 가능한 context를 초과할 수 있습니다.
- **계산 비용**: 큰 context를 처리하는 것은 비용이 큽니다. Attention mechanism은 제곱에 비례하여 커집니다.
- **반복 패턴**: 대화마다 비슷한 작업을 할 때 context를 매번 다시 설명해야 합니다.
- **정보 손실**: Context가 가득 차면 앞선 중요한 정보를 잃습니다.

### 해결책

Claude Sonnet 4.6은 두 가지 강력한 기능을 도입합니다.

1. **Memory Tool** (`memory_20250818`): 대화 간 학습을 가능하게 합니다.
   - Claude는 배운 내용을 향후 참조를 위해 기록할 수 있습니다.
   - `/memories` 디렉터리 아래의 파일 기반 시스템
   - Client-side 구현으로 완전한 제어 가능

**지원 모델**: Claude Opus 4.1 (`claude-opus-4-1`), Claude Opus 4 (`claude-opus-4`), Claude Sonnet 4.6 (`claude-sonnet-4-6`), Claude Sonnet 4 (`claude-sonnet-4`), Claude Haiku 4.5 (`claude-haiku-4-5`)

### 이점

**시간이 지나면서 특정 작업을 더 잘하는** AI agent를 구축하세요.

- **Session 1**: Claude가 문제를 해결하고 패턴을 기록합니다.
- **Session 2**: Claude가 학습한 패턴을 즉시 적용합니다(더 빠릅니다!).
- **긴 session**: Context editing으로 대화를 관리 가능한 상태로 유지합니다.

사람처럼 메모를 작성하고 다시 참조할 notebook을 Claude에 준다고 생각하세요.


## 2. 사용 사례 {#use-cases}

Memory와 context 관리는 강력한 새 workflow를 가능하게 합니다.

### 🔍 Code Review Assistant
- 이전 review에서 debugging 패턴 학습
- 향후 session에서 비슷한 bug를 즉시 인식
- 팀별 코드 품질 지식 구축
- **Production 준비 완료**: GitHub PR review를 위해 [claude-code-action](https://github.com/anthropics/claude-code-action)과 통합

### 📚 연구 Assistant
- 여러 session에 걸쳐 주제에 대한 지식 축적
- 서로 다른 연구 thread의 통찰 연결
- 참고 문헌과 출처 추적 유지

### 💬 고객 지원 Bot
- 사용자 선호와 소통 방식 학습
- 일반적인 문제와 해결책 기억
- 상호작용을 통해 제품 knowledge base 구축

### 📊 데이터 분석 Helper
- Dataset 패턴과 이상 징후 기억
- 잘 작동하는 분석 기법 저장
- 시간에 따라 도메인별 통찰 구축

**지원 모델**: Claude Opus 4.1 (`claude-opus-4-1`), Claude Sonnet 4.6 (`claude-sonnet-4-6`)

Memory(패턴 학습)와 context editing(긴 review 처리)을 모두 명확히 보여주므로 **이 cookbook은 Code Review Assistant에 초점을 맞춥니다**.


## 3. 빠른 시작 예제 {#quick-start}

간단한 예제로 memory와 context 관리가 동작하는 모습을 살펴보겠습니다.


### 설정

먼저 dependencies를 설치하고 환경을 설정합니다.


In [1]:
%%capture
# 필요한 package 설치
# 옵션 1: requirements.txt에서 설치
# %pip install -q -r requirements.txt

# 옵션 2: 직접 설치
%pip install -q anthropic python-dotenv ipykernel

**⚠️ 중요**: 이 디렉터리에 `.env` 파일을 만드세요.

```bash
# .env.example을 .env로 복사하고 API key 추가
cp .env.example .env
```

그런 다음 `.env`를 편집하여 https://console.anthropic.com/ 에서 발급받은 Anthropic API key를 추가합니다.


In [2]:
import os
from typing import cast

from anthropic import Anthropic
from dotenv import load_dotenv

# 환경 변수 불러오기
load_dotenv()

# 모델 설정 - 자동 업데이트를 위해 alias 사용
MODEL = "claude-sonnet-4-6"  # ANTHROPIC_MODEL 환경 변수로 재정의 가능
if os.getenv("ANTHROPIC_MODEL"):
    MODEL = os.getenv("ANTHROPIC_MODEL")

if not API_KEY:
    raise ValueError("ANTHROPIC_API_KEY not found. Copy .env.example to .env and add your API key.")

if not MODEL:
    raise ValueError("ANTHROPIC_MODEL not found. Copy .env.example to .env and set the model.")

MODEL = cast(str, MODEL)

client = Anthropic(api_key=API_KEY)

print("✓ API key loaded")
print(f"✓ Using model: {MODEL}")

✓ API key loaded
✓ Using model: claude-sonnet-4-6


### 예제 1: 기본 Memory 사용

Claude가 향후 참조할 정보를 저장하기 위해 memory를 사용하는 모습을 살펴보겠습니다.


**Helper 함수**

이 예제는 `demo_helpers.py`의 helper 함수를 사용합니다.

- **`run_conversation_loop()`**: API 대화 loop 처리
  - memory tool을 활성화하여 Claude API 호출
  - tool use 실행(memory 작업)
  - Claude가 tool 사용을 멈출 때까지 계속
  - 최종 응답 반환

- **`run_conversation_turn()`**: 단일 turn(예제 3에서 사용)
  - 위와 같지만 API 호출 한 번 후 반환
  - 세밀한 제어가 필요할 때 유용

- **`print_context_management_info()`**: context clearing 통계 표시
  - 절약된 token과 지워진 tool use 표시
  - Context editing 실행 시점의 시각화에 도움


**⚠️ Memory 지우기에 관한 참고**

다음 cell은 이 시연을 빈 상태에서 시작하기 위해 모든 memory 파일을 지웁니다. Notebook을 여러 번 실행하면서 일관된 결과를 확인할 때 유용합니다.

**Production 애플리케이션에서는** 학습한 패턴이 영구적으로 제거되므로 모든 memory를 지울지 신중히 고려해야 합니다. 대신 선택적 삭제를 사용하거나 memory를 프로젝트별 디렉터리로 구성하는 것을 고려하세요.


In [3]:
# Helper 함수 import
from memory_demo.demo_helpers import (
    run_conversation_loop,
    run_conversation_turn,
    print_context_management_info,
)
from memory_tool import MemoryToolHandler

# 초기화
client = Anthropic()
memory = MemoryToolHandler(base_path="./demo_memory")

# 새로 시작하기 위해 기존 memory 모두 지우기
print("🧹 Clearing previous memories...")
memory.clear_all_memory()
print("✓ Memory cleared\n")

# Race condition bug가 있는 예제 코드 불러오기
with open("memory_demo/sample_code/web_scraper_v1.py", "r") as f:
    code_to_review = f.read()

messages = [
    {
        "role": "user",
        "content": f"I'm reviewing a multi-threaded web scraper that sometimes returns fewer results than expected. The count is inconsistent across runs. Can you find the issue?\n\n```python\n{code_to_review}\n```",
    }
]

print("=" * 60)
print("📝 SESSION 1: Learning from a bug")
print("=" * 60)

# 대화 loop 실행
response = run_conversation_loop(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    max_tokens=2048,
    max_turns=5,
    verbose=True,
)

print("\n" + "=" * 60)
print("✅ Session 1 complete!")
print("=" * 60)

🧹 Clearing previous memories...
✓ Memory cleared

📝 SESSION 1: Learning from a bug

🔄 Turn 1:
💬 Claude: I'll review this code for the race condition issue. Let me first check my memory, then analyze the problem.

  🔧 Memory tool: view /memories
  ✓ Result: Directory: /memories
(empty)

🔄 Turn 2:
  🔧 Memory tool: create /memories/review.md
  ✓ Result: File created successfully at /memories/review.md

🔄 Turn 3:
💬 Claude: ## Code Review: Multi-threaded Web Scraper Race Condition

### 🔴 **Critical Issue Found: Race Condition in Shared State**

You've correctly identified the problem! The code has **race conditions** caused by multiple threads modifying shared lists without synchronization.

---

### **The Problem**

**Lines with race conditions:**
```python
self.results.append(result)  # RACE CONDITION
```

**Problem:** 
- Python's `list.append()` is **NOT thread-safe** for concurrent modifications
- Multiple threads simultaneously appending can cause:
  - **Lost updates:** One thread's ap

**무슨 일이 일어났나요?**

1. Claude가 memory를 확인했습니다(최초 실행 시 비어 있음).
2. Bug를 파악했습니다: **race condition** - 여러 thread가 synchronization 없이 shared state(`self.results`, `self.failed_urls`)를 수정합니다.
3. 향후 참조를 위해 memory에 동시성 패턴을 저장했습니다.

이제 Claude가 **새 대화**에서 학습한 패턴을 적용하는 마법을 살펴보겠습니다.


### 예제 2: 대화 간 학습

완전히 새로운 대화를 시작해도 memory는 유지됩니다!


In [4]:
# 새 대화(빈 messages)
# 비슷한 동시성 문제가 있는 API client 코드 불러오기
with open("memory_demo/sample_code/api_client_v1.py", "r") as f:
    code_to_review = f.read()

messages = [
    {"role": "user", "content": f"Review this API client code:\n\n```python\n{code_to_review}\n```"}
]

print("=" * 60)
print("🚀 SESSION 2: Applying learned pattern")
print("=" * 60)

# 대화 loop 실행
response = run_conversation_loop(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    max_tokens=2048,
    max_turns=5,
    verbose=True,
)

print("\n" + "=" * 60)
print("✅ Session 2 complete!")
print("=" * 60)

🚀 SESSION 2: Applying learned pattern

🔄 Turn 1:
  🔧 Memory tool: view /memories
  ✓ Result: Directory: /memories
- review.md

🔄 Turn 2:
  🔧 Memory tool: view /memories/review.md
  ✓ Result:    1: # Code Review: Multi-threaded Web Scraper
   2: 
   3: ## Issue
   4: Revi...

🔄 Turn 3:
  🔧 Memory tool: str_replace /memories/review.md
  ✓ Result: File /memories/review.md has been edited successfully

🔄 Turn 4:
💬 Claude: ## Code Review: Async API Client

### Summary
This code has **concurrency issues** related to shared mutable state being accessed from multiple coroutines without proper synchronization. While the author correctly identifies these as bugs in comments, let me provide a comprehensive review.

However, the actual race condition risk here is **somewhat overstated** because:
- List `.append()` is atomic in CPython
- The `+=` operation on integers is also atomic
- Coroutines only switch at `await` points, and there are none between the operations

**BUT** this is still problema

**차이점을 확인하세요.**

- Claude가 **즉시 memory를 확인**하여 thread-safety/동시성 패턴을 찾았습니다.
- 다시 학습하지 않고 async 코드의 비슷한 문제를 **즉시** 인식했습니다.
- Shared mutable state에 대해 저장한 지식을 적용하여 응답이 **더 빨랐습니다**.

이것이 실제 **대화 간 학습**입니다!


### 예제 3: Memory를 보존하면서 Context 지우기

코드 파일이 많은 **긴 review session**에서는 무슨 일이 일어날까요?

- 이전 review의 tool 결과로 context가 가득 찹니다.
- 하지만 memory(학습한 패턴)는 유지되어야 합니다!

**Context editing**을 실행하여 Claude가 이를 자동 관리하는 방식을 살펴보겠습니다.

**설정에 관한 참고:** Memory tool 작업의 결과가 작으므로(각각 약 50~150 token) `clear_at_least: 50` token을 사용합니다. Web search나 code execution처럼 더 큰 tool 결과를 다루는 production에서는 3000~5000 token 같은 더 높은 값을 사용합니다.


In [5]:
# 두 지우기 전략 모두로 context 관리 설정
# Demo용 낮은 threshold - production에서는 30~40k token 사용
CONTEXT_MANAGEMENT = {
    "edits": [
        # 전략을 조합할 때 thinking management가 먼저 와야 합니다
        {
            "type": "clear_thinking_20251015",
            "keep": {"type": "thinking_turns", "value": 1}  # 마지막 turn의 thinking만 유지
        },
        {
            "type": "clear_tool_uses_20250919",
            "trigger": {"type": "input_tokens", "value": 5000},  # Demo용 낮은 threshold
            "keep": {"type": "tool_uses", "value": 2},  # 마지막 tool use 2개 유지
            "clear_at_least": {"type": "input_tokens", "value": 2000}
        }
    ]
}

# Extended thinking 설정(clear_thinking 전략에 필요)
THINKING = {
    "type": "enabled",
    "budget_tokens": 1024  # Turn당 thinking budget
}

# 이전 session에서 계속 - memory가 유지됩니다!
print("=" * 60)
print("📚 SESSION 3: Long review session with context clearing")
print("=" * 60)
print()

# Messages 정리 - 이전 session의 빈 content 제거
# 이를 통해 계속할 수 있는 유효한 message state를 확보합니다
cleaned_messages = []
for msg in messages:
    if isinstance(msg.get("content"), list):
        # 빈 content block filtering
        content = [c for c in msg["content"] if c]
        if content:
            cleaned_messages.append({"role": msg["role"], "content": content})
    elif msg.get("content"):
        cleaned_messages.append(msg)

messages = cleaned_messages

# Review 1: Data processor(더 큰 파일)
with open("memory_demo/sample_code/data_processor_v1.py", "r") as f:
    data_processor_code = f.read()

messages.append({
    "role": "user",
    "content": f"Review this data processor:\n\n```python\n{data_processor_code}\n```"
})

print("📝 Review 1: Data processor")
response = run_conversation_turn(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    context_management=CONTEXT_MANAGEMENT,
    thinking=THINKING,
    max_tokens=4096,
    verbose=True
)

# Messages에 응답 추가
messages.append({"role": "assistant", "content": response[1]})
if response[2]:
    messages.append({"role": "user", "content": response[2]})

print(f"  📊 Input tokens: {response[0].usage.input_tokens:,}")
context_cleared, saved = print_context_management_info(response[0])
print()

# Review 2: SQL 코드
with open("memory_demo/sample_code/sql_query_builder.py", "r") as f:
    sql_code = f.read()

messages.append({
    "role": "user",
    "content": f"Review this SQL query builder:\n\n```python\n{sql_code}\n```"
})

print("📝 Review 2: SQL query builder")
response = run_conversation_turn(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    context_management=CONTEXT_MANAGEMENT,
    thinking=THINKING,
    max_tokens=4096,
    verbose=True
)

messages.append({"role": "assistant", "content": response[1]})
if response[2]:
    messages.append({"role": "user", "content": response[2]})

print(f"  📊 Input tokens: {response[0].usage.input_tokens:,}")
context_cleared, saved = print_context_management_info(response[0])
print()

# Review 3: 지우기가 실행되도록 review 하나 더 추가
with open("memory_demo/sample_code/web_scraper_v1.py", "r") as f:
    scraper_code = f.read()

messages.append({
    "role": "user",
    "content": f"Quick check - any issues here?\n\n```python\n{scraper_code}\n```"
})

print("📝 Review 3: Web scraper (should trigger clearing)")
response = run_conversation_turn(
    client=client,
    model=MODEL,
    messages=messages,
    memory_handler=memory,
    system="You are a code reviewer.",
    context_management=CONTEXT_MANAGEMENT,
    thinking=THINKING,
    max_tokens=4096,
    verbose=True
)

messages.append({"role": "assistant", "content": response[1]})
if response[2]:
    messages.append({"role": "user", "content": response[2]})

print(f"  📊 Input tokens: {response[0].usage.input_tokens:,}")
context_cleared, saved = print_context_management_info(response[0])

print()
print("=" * 60)
print("✅ Session 3 complete!")
print("=" * 60)

📚 SESSION 3: Long review session with context clearing

📝 Review 1: Data processor
🧠 Thinking: The user wants me to review a data processor with multiple concurrency and thread-safety issues. Let...
  🔧 Memory tool: str_replace /memories/review.md
  ✓ Result: File /memories/review.md has been edited successfully
  📊 Input tokens: 6,611
  ℹ️  Context below threshold - no clearing triggered

📝 Review 2: SQL query builder
🧠 Thinking: The user is now asking me to review a SQL query builder with SQL injection vulnerabilities. This is ...
  🔧 Memory tool: str_replace /memories/review.md
  ✓ Result: File /memories/review.md has been edited successfully
  📊 Input tokens: 7,923
  ✂️  Context editing triggered!
      • Cleared 1 thinking turn(s), saved 166 tokens
      • After clearing: 7,923 tokens

📝 Review 3: Web scraper (should trigger clearing)
🧠 Thinking: This is a quick check request for a web scraper with threading issues. Let me quickly identify the p...
  🔧 Memory tool: str_replace /me

**방금 무슨 일이 일어났나요?**

Extended thinking을 활성화한 여러 review 중 context가 커지면서 context editing이 적용되었습니다.
1. **Thinking block 지우기** - 이전 turn의 오래된 thinking을 먼저 제거
2. **Tool 결과 지우기** - Threshold 초과 시 오래된 memory tool 결과 제거
3. **Memory 파일 유지** - Claude는 여전히 학습한 패턴 조회 가능
4. **Token 사용량 관리** - Thinking과 tool 결과 모두에서 token 절약

이는 핵심 이점을 보여줍니다.
- **단기 memory**(대화 context + thinking) → 공간 절약을 위해 지움
- **장기 memory**(저장된 패턴) → Session 간 유지

지운 후에도 memory가 유지되는지 확인하겠습니다.


In [6]:
# Context 지우기 후 memory 유지 확인
import os

print("📂 Memory files in demo_memory/:")
print()

for root, dirs, files in os.walk("./demo_memory"):
    # 표시용 상대 경로 계산
    level = root.replace("./demo_memory", "").count(os.sep)
    indent = "  " * level
    folder_name = os.path.basename(root) or "demo_memory"
    print(f"{indent}{folder_name}/")

    sub_indent = "  " * (level + 1)
    for file in files:
        file_path = os.path.join(root, file)
        size = os.path.getsize(file_path)
        print(f"{sub_indent}├── {file} ({size} bytes)")

print()
print("✅ All learned patterns preserved despite context clearing!")

📂 Memory files in demo_memory/:

demo_memory/
  memories/
    ├── review.md (318 bytes)

✅ All learned patterns preserved despite context clearing!


## 4. 동작 방식 {#how-it-works}

### Memory Tool 아키텍처

Memory tool은 **client-side**이며 저장소를 직접 제어합니다. Claude가 tool을 호출하고 애플리케이션이 실행합니다.

#### Memory Tool 명령

| 명령 | 설명 | 예제 |
|---------|-------------|---------|
| `view` | 디렉터리 또는 파일 내용 표시 | `{"command": "view", "path": "/memories"}` |
| `create` | 파일 생성 또는 덮어쓰기 | `{"command": "create", "path": "/memories/notes.md", "file_text": "..."}` |
| `str_replace` | 파일의 텍스트 교체 | `{"command": "str_replace", "path": "...", "old_str": "...", "new_str": "..."}` |
| `insert` | 줄 번호에 텍스트 삽입 | `{"command": "insert", "path": "...", "insert_line": 2, "insert_text": "..."}` |
| `delete` | 파일 또는 디렉터리 삭제 | `{"command": "delete", "path": "/memories/old.txt"}` |
| `rename` | 파일 이름 변경 또는 이동 | `{"command": "rename", "old_path": "...", "new_path": "..."}` |

경로 검증과 보안 조치가 포함된 전체 구현은 `memory_tool.py`를 참조하세요.


### Thinking Management (`clear_thinking_20251015`)

Extended thinking을 사용하면 thinking block이 쌓여 token을 소모합니다. `clear_thinking` 전략은 이를 자동 관리합니다.

**중요**: 이 전략은 API 호출에서 `thinking`을 활성화해야 합니다.

**API 호출 패턴**(extended thinking 활성화):

```python
response = client.beta.messages.create(
    betas=["context-management-2025-06-27"],  # 필수 beta flag
    model="claude-sonnet-4-6",
    messages=messages,
    tools=[{"type": "memory_20250818", "name": "memory"}],
    thinking={"type": "enabled", "budget_tokens": 10000},  # Thinking 활성화
    context_management={  # Context editing 설정
        "edits": [
            {
                "type": "clear_thinking_20251015",
                "keep": {"type": "thinking_turns", "value": 1}  # 마지막 turn만 유지
            },
            {
                "type": "clear_tool_uses_20250919",
                "trigger": {"type": "input_tokens", "value": 35000},
                "keep": {"type": "tool_uses", "value": 5}
            }
        ]
    },
    max_tokens=2048
)
```

**핵심 사항:**
- 전략을 조합할 때 `clear_thinking`이 **먼저** 와야 합니다.
- Extended thinking을 활성화해야 합니다(`thinking={"type": "enabled", ...}`).
- Cache hit를 최대화하려면 `"keep": "all"`로 모든 thinking block을 보존하세요.
- Thinking의 trigger는 선택 사항입니다(`keep` 값에 따라 지움).


### Demo 코드 이해하기

`code_review_demo.py`의 핵심 구현 세부사항:

```python
class CodeReviewAssistant:
    def __init__(self, memory_storage_path="./memory_storage"):
        self.client = Anthropic()
        self.memory_handler = MemoryToolHandler(base_path=memory_storage_path)
        self.messages = []
    
    def review_code(self, code, filename, description=""):
        # 1. User message 추가
        self.messages.append({...})
        
        # 2. Tool 실행이 있는 대화 loop
        while True:
            response = self.client.beta.messages.create(
                model=MODEL,
                system=self._create_system_prompt(),
                messages=self.messages,
                tools=[{"type": "memory_20250818", "name": "memory"}],
                betas=["context-management-2025-06-27"],
                context_management=CONTEXT_MANAGEMENT
            )
            
            # 3. Tool use 실행
            tool_results = []
            for content in response.content:
                if content.type == "tool_use":
                    result = self._execute_tool_use(content)
                    tool_results.append({...})
            
            # 4. Tool use가 있으면 계속하고, 없으면 완료
            if tool_results:
                self.messages.append({"role": "user", "content": tool_results})
            else:
                break
```

**핵심 패턴**: Tool use가 있는 동안 API를 계속 호출하고 실행하여 결과를 돌려줍니다.


### Claude가 실제로 학습하는 것

Memory를 강력하게 하는 것은 단순한 문법이 아닌 **semantic pattern recognition**입니다.

**Session 1: Thread 기반 Web Scraper**

```python
# Bug: Race condition
class WebScraper:
    def __init__(self):
        self.results = []  # Shared state!
    
    def scrape_urls(self, urls):
        with ThreadPoolExecutor() as executor:
            for future in as_completed(futures):
                self.results.append(future.result())  # RACE!
```

**Claude가 Memory에 저장하는 것**(예제 파일: `/memories/concurrency_patterns/thread_safety.md`):

Claude는 이 패턴을 마주하면 다음 통찰을 memory 파일에 저장합니다.
- **증상**: 동시 작업에서 일관되지 않은 결과
- **원인**: 여러 thread가 shared mutable state(list/dict)를 수정
- **해결책**: Lock, thread-safe 자료구조를 사용하거나 대신 결과 반환
- **위험 징후**: Thread callback의 instance 변수, 사용하지 않는 lock, counter 증가

---

**Session 2: Async API Client**(새 대화!)

Claude는 먼저 memory를 확인하여 thread-safety 패턴을 찾은 다음:
1. Async 코드에서 비슷한 패턴을 **인식**합니다(coroutine도 interleave될 수 있음).
2. 해결책을 즉시 **적용**합니다(재학습 불필요).
3. 저장된 지식을 참조하여 **설명**합니다.

```python
# Claude는 이를 즉시 발견합니다:
async def fetch_all(self, endpoints):
    for coro in asyncio.as_completed(tasks):
        self.responses.append(await coro)  # 같은 패턴!
```

---

**중요한 이유:**

- ❌ **Syntax checker**는 race condition을 완전히 놓칩니다.
- ✅ **Claude는 학습한** 아키텍처 패턴을 여러 context에 적용합니다.
- ✅ **언어 간 적용**: Go, Java, Rust 동시성에도 패턴을 적용할 수 있습니다.
- ✅ **개선됨**: Review마다 knowledge base에 지식이 추가됩니다.


### 예제 코드 파일

Demo는 이 예제 파일을 사용합니다(모두 동시성/thread-safety bug가 있음).

- `memory_demo/sample_code/web_scraper_v1.py` - Race condition: thread가 shared state 수정
- `memory_demo/sample_code/api_client_v1.py` - Async context의 비슷한 동시성 bug
- `memory_demo/sample_code/data_processor_v1.py` - 긴 session demo를 위한 여러 동시성 문제

하나를 살펴보겠습니다.


**`memory_demo/sample_code/web_scraper_v1.py`**

```python
"""
Race condition bug가 있는 동시 web scraper.
여러 thread가 synchronization 없이 shared state를 수정합니다.
"""

import time
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Dict

import requests


class WebScraper:
    """여러 URL을 동시에 조회하는 web scraper."""

    def __init__(self, max_workers: int = 10):
        self.max_workers = max_workers
        self.results = []  # BUG: 여러 thread가 shared mutable state에 접근합니다!
        self.failed_urls = []  # BUG: 또 다른 race condition!

    def fetch_url(self, url: str) -> Dict[str, any]:
        """URL 하나를 조회하여 결과를 반환합니다."""
        try:
            response = requests.get(url, timeout=5)
            response.raise_for_status()
            return {
                "url": url,
                "status": response.status_code,
                "content_length": len(response.content),
            }
        except requests.exceptions.RequestException as e:
            return {"url": url, "error": str(e)}

    def scrape_urls(self, urls: List[str]) -> List[Dict[str, any]]:
        """
        여러 URL을 동시에 scrape합니다.

        BUG: 여러 thread가 locking 없이 self.results에 접근합니다!
        이로 인해 결과가 손실되거나 손상될 수 있는 race condition이 발생합니다.
        """
        with ThreadPoolExecutor(max_workers=self.max_workers) as executor:
            futures = [executor.submit(self.fetch_url, url) for url in urls]

            for future in as_completed(futures):
                result = future.result()

                # RACE CONDITION: 여러 thread가 동시에 self.results에 추가합니다
                if "error" in result:
                    self.failed_urls.append(result["url"])  # RACE CONDITION
                else:
                    self.results.append(result)  # RACE CONDITION

        return self.results
```


**Bug**: 여러 thread가 locking 없이 `self.results`와 `self.failed_urls`를 수정합니다!

Claude는 다음을 수행합니다.
1. Race condition 식별
2. `/memories/concurrency_patterns/thread_safety.md`에 패턴 저장
3. Session 2에서 이 동시성 패턴을 async 코드에 적용


### Demo 개요

완전한 Code Review Assistant를 구축했습니다. 구현은 `memory_demo/code_review_demo.py`에 있습니다.

**대화형 demo 실행:**
```bash
python memory_demo/code_review_demo.py
```

Demo는 다음을 보여줍니다.
1. **Session 1**: Bug가 있는 Python 코드 review → Claude가 패턴 학습
2. **Session 2**: 비슷한 코드 review(새 대화) → Claude가 패턴 적용
3. **Session 3**: 긴 review session → Context editing으로 관리 가능한 상태 유지


## 7. 모범 사례와 보안 {#best-practices}

### Memory 관리

**해야 할 것:**
- ✅ 대화 기록 대신 작업과 관련된 패턴 저장
- ✅ 명확한 디렉터리 구조로 구성
- ✅ 의미가 분명한 파일 이름 사용
- ✅ 주기적으로 memory 검토 및 정리

**하지 말아야 할 것:**
- ❌ 민감한 정보(password, API key, PII) 저장
- ❌ Memory가 제한 없이 커지도록 방치
- ❌ 모든 것을 무분별하게 저장

### 보안: Path Traversal 보호

**중요**: Directory traversal 공격을 방지하려면 항상 경로를 검증하세요. 구현은 `memory_tool.py`를 참조하세요.

### 보안: Memory Poisoning

**⚠️ 중대한 위험**: Memory 파일은 Claude의 context로 다시 읽히므로 prompt injection의 잠재적 공격 경로가 됩니다.

**완화 전략:**
1. **Content Sanitization**: 저장 전 위험한 패턴 filtering
2. **Memory 범위 분리**: 사용자별/프로젝트별 분리
3. **Memory Auditing**: 모든 memory 작업의 log 기록과 scan
4. **Prompt Engineering**: Claude에 memory 안의 지시를 무시하도록 지시

전체 보안 구현은 `memory_tool.py`, 테스트는 `tests/`를 참조하세요.


## 결론

### 달성한 것

이 cookbook에서 다음을 배웠습니다.
- ✅ 대화 간 학습을 위한 **memory tool 구현**(Session 1과 2에서 패턴 인식이 유지됨을 보여줌)
- ✅ Token trigger와 보존 정책을 사용한 **context editing 설정**(Session 3에서 자동 지우기를 시연)
- ✅ 경로 검증과 memory poisoning 방지를 포함한 **보안 모범 사례 적용**

### 패턴 적용

**프로젝트에 적용:**
1. 패턴을 위한 단일 memory 파일로 시작(예: `/memories/patterns.md`)
2. Production 사용 시 context editing trigger를 30~40k token으로 설정
3. 교차 오염을 방지하기 위해 프로젝트별 memory 분리 구현

**다른 애플리케이션:**
- **고객 지원**: 사용자 선호와 일반적인 문제 해결책 저장
- **연구 assistant**: Session 간 도메인 지식 축적
- **데이터 분석**: Dataset 특성과 성공적인 기법 기억

### 다음 단계

- **Production 배포**: GitHub PR review에 [claude-code-action](https://github.com/anthropics/claude-code-action) 사용
- **보안 강화**: `memory_tool.py`의 memory poisoning 완화 방법 검토
- **Extended thinking**: 계산이 많이 필요한 작업을 위한 thinking management 탐색

### 자료

- [Memory tool 문서](https://docs.claude.com/en/docs/agents-and-tools/tool-use/memory-tool)
- [Claude API reference](https://docs.claude.com/en/api/messages)
- [지원](https://support.claude.com)

Memory와 context 관리는 **beta**입니다. 개선할 수 있도록 feedback을 공유해 주세요!
